First Best

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC, NuSVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

# 1. Chargement des données
df = pd.read_csv("parkinson.csv")
X = df.drop(columns=['ID', 'Recording', 'Status'])
y = df['Status']
X = pd.get_dummies(X, drop_first=True)

# 2. Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Définir les classifieurs
classifiers = {
    "Naïve Bayes": GaussianNB(),
    "c-SVM": SVC(kernel="rbf", C=1.0),
    "nu-SVM": NuSVC(nu=0.5, kernel="rbf"),
    "MLP": MLPClassifier(max_iter=2000),
    "KNN": KNeighborsClassifier(),
    "Random Forest": RandomForestClassifier()
}

# 4. Sélection de caractéristiques avec RF comme estimateur de base
base_selector = RandomForestClassifier()
sfs = SequentialFeatureSelector(base_selector, n_features_to_select='auto', direction='forward', cv=5, n_jobs=-1)
X_selected = sfs.fit_transform(X_scaled, y)

# 5. Évaluation par validation croisée 10-fold
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

results = {}

for name, clf in classifiers.items():
    acc = cross_val_score(clf, X_selected, y, cv=cv, scoring='accuracy').mean()
    pre = cross_val_score(clf, X_selected, y, cv=cv, scoring='precision').mean()
    rec = cross_val_score(clf, X_selected, y, cv=cv, scoring='recall').mean()
    f1 = cross_val_score(clf, X_selected, y, cv=cv, scoring='f1').mean()
    results[name] = [acc, pre, rec, f1]
    print(f"{name}: Accuracy={acc:.3f}, Precision={pre:.3f}, Recall={rec:.3f}, F1-score={f1:.3f}")

# 6. Affichage des résultats
metrics = ["Accuracy", "Precision", "Recall", "F1-score"]
df_results = pd.DataFrame(results, index=metrics).T
print("\nRésultats avec Wrapper-based PSO (base KNN):")
display(df_results)

# 7. Tracé des résultats
df_results.plot(kind='bar', figsize=(10,6))
plt.title("Wrapper-based FS (RF) with First Best — Classifier Comparison")
plt.ylabel("Score")
plt.ylim(0.6, 0.95)
plt.grid(axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


Greed StepWise

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC, NuSVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

# 1. Chargement des données
df = pd.read_csv("parkinson.csv")
X = df.drop(columns=['ID', 'Recording', 'Status'])
y = df['Status']
X = pd.get_dummies(X, drop_first=True)

# 2. Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Définir les classifieurs
classifiers = {
    "Naïve Bayes": GaussianNB(),
    "c-SVM": SVC(kernel="rbf", C=1.0),
    "nu-SVM": NuSVC(nu=0.5, kernel="rbf"),
    "MLP": MLPClassifier(max_iter=2000),
    "KNN": KNeighborsClassifier(),
    "Random Forest": RandomForestClassifier()
}

# 4. Sélection de caractéristiques avec RF comme estimateur de base
base_selector = RandomForestClassifier()
sfs = SequentialFeatureSelector(base_selector, n_features_to_select='auto', direction='backward', cv=5, n_jobs=-1)
X_selected = sfs.fit_transform(X_scaled, y)

# 5. Évaluation par validation croisée 10-fold
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

results = {}

for name, clf in classifiers.items():
    acc = cross_val_score(clf, X_selected, y, cv=cv, scoring='accuracy').mean()
    pre = cross_val_score(clf, X_selected, y, cv=cv, scoring='precision').mean()
    rec = cross_val_score(clf, X_selected, y, cv=cv, scoring='recall').mean()
    f1 = cross_val_score(clf, X_selected, y, cv=cv, scoring='f1').mean()
    results[name] = [acc, pre, rec, f1]

# 6. Affichage des résultats
metrics = ["Accuracy", "Precision", "Recall", "F1-score"]
df_results = pd.DataFrame(results, index=metrics).T
print("\nRésultats avec Wrapper-based PSO (base KNN):")
display(df_results)

# 7. Tracé des résultats
df_results.plot(kind='bar', figsize=(10,6))
plt.title("Wrapper-based FS (RF) with Greed StepWise — Classifier Comparison")
plt.ylabel("Score")
plt.ylim(0.6, 0.95)
plt.grid(axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


PSO

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC, NuSVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

import pyswarms as ps

# 1. Charger et préparer les données
df = pd.read_csv("parkinson.csv")
X = df.drop(columns=['ID', 'Recording', 'Status'])
y = df['Status']
X = pd.get_dummies(X, drop_first=True)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
n_features = X_scaled.shape[1]

# 2. Fonction objectif (RF comme base classifier)
def objective_function(particles):
    scores = []
    for particle in particles:
        mask = particle.astype(bool)
        if np.sum(mask) == 0:
            scores.append(1.0)  # pénalité maximale
            continue
        X_subset = X_scaled[:, mask]
        clf = RandomForestClassifier()
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        score = cross_val_score(clf, X_subset, y, cv=cv, scoring='accuracy').mean()
        scores.append(1 - score)  # minimiser 1 - accuracy
    return np.array(scores)

# 3. PSO - Feature Selection
options = {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}
optimizer = ps.discrete.BinaryPSO(n_particles=20, dimensions=n_features, options=options)
cost, best_position = optimizer.optimize(objective_function, iters=30, verbose=True)

# 4. Sélection finale des caractéristiques
selected_features = best_position.astype(bool)
X_selected = X_scaled[:, selected_features]

# 5. Évaluation des classifieurs avec validation croisée
classifiers = {
    "Naïve Bayes": GaussianNB(),
    "c-SVM": SVC(kernel="rbf"),
    "nu-SVM": NuSVC(nu=0.5, kernel="rbf"),
    "MLP": MLPClassifier(max_iter=2000),
    "KNN": KNeighborsClassifier(),
    "Random Forest": RandomForestClassifier()
}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
results = {}

for name, clf in classifiers.items():
    acc = cross_val_score(clf, X_selected, y, cv=cv, scoring='accuracy').mean()
    pre = cross_val_score(clf, X_selected, y, cv=cv, scoring='precision').mean()
    rec = cross_val_score(clf, X_selected, y, cv=cv, scoring='recall').mean()
    f1 = cross_val_score(clf, X_selected, y, cv=cv, scoring='f1').mean()
    results[name] = [acc, pre, rec, f1]

# 6. Visualisation
metrics = ["Accuracy", "Precision", "Recall", "F1-score"]
df_results = pd.DataFrame(results, index=metrics).T
print("\nRésultats avec Wrapper-based PSO (base KNN):")
display(df_results)

df_results.plot(kind='bar', figsize=(10,6))
plt.title("Wrapper-based FS (PSO + RF) — Classifier Comparison")
plt.ylabel("Score")
plt.ylim(0.6, 0.95)
plt.grid(axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
